In [1]:
# library(MASS)     # para fitdistr e binomial negativa
library(tidyverse)
library(dplyr)
# library(ggplot2)
# library(stringr)
# library(tidyr)
# library(purrr)
# library(scales)
# library(fitdistrplus)  # para ajustar e comparar distribuições
# library(broom)
library(lme4)
# library(broom.mixed)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.3     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   4.0.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.0
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack




In [2]:
dados = read_csv("/home/aninha/Desktop/Doutorado/Dados/gabarito_foco_ausente.csv")
head(dados)
glimpse(dados)

Rows: 80480 Columns: 9
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (6): cd_exame, faixa_etaria, regiao_sp, sexo, origem, dente_col
dbl (3): idade, ausente, dente

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


cd_exame,faixa_etaria,regiao_sp,idade,sexo,origem,dente_col,ausente,dente
<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>
alp.026080,20-44,Oeste,24,masculino,Alphaville,ausente_11,0,11
alp.026080,20-44,Oeste,24,masculino,Alphaville,ausente_12,0,12
alp.026080,20-44,Oeste,24,masculino,Alphaville,ausente_13,0,13
alp.026080,20-44,Oeste,24,masculino,Alphaville,ausente_14,0,14
alp.026080,20-44,Oeste,24,masculino,Alphaville,ausente_15,0,15
alp.026080,20-44,Oeste,24,masculino,Alphaville,ausente_16,0,16


Rows: 80,480
Columns: 9
$ cd_exame     <chr> "alp.026080", "alp.026080", "alp.026080", "alp.026080", "…
$ faixa_etaria <chr> "20-44", "20-44", "20-44", "20-44", "20-44", "20-44", "20…
$ regiao_sp    <chr> "Oeste", "Oeste", "Oeste", "Oeste", "Oeste", "Oeste", "Oe…
$ idade        <dbl> 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 2…
$ sexo         <chr> "masculino", "masculino", "masculino", "masculino", "masc…
$ origem       <chr> "Alphaville", "Alphaville", "Alphaville", "Alphaville", "…
$ dente_col    <chr> "ausente_11", "ausente_12", "ausente_13", "ausente_14", "…
$ ausente      <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, …
$ dente        <dbl> 11, 12, 13, 14, 15, 16, 17, 18, 21, 22, 23, 24, 25, 26, 2…


In [3]:
dados$faixa_etaria = factor(dados$faixa_etaria, ordered = FALSE)
dados$regiao_sp = factor(dados$regiao_sp, ordered = FALSE)
dados$sexo = factor(dados$sexo, ordered = FALSE)
dados$origem = factor(dados$origem, ordered = FALSE)
dados$dente = factor(dados$dente, ordered = FALSE)

dados$idade = as.integer(dados$idade)
dados$ausente = as.integer(dados$ausente)

In [4]:
glimpse(dados)

Rows: 80,480
Columns: 9
$ cd_exame     <chr> "alp.026080", "alp.026080", "alp.026080", "alp.026080", "…
$ faixa_etaria <fct> 20-44, 20-44, 20-44, 20-44, 20-44, 20-44, 20-44, 20-44, 2…
$ regiao_sp    <fct> Oeste, Oeste, Oeste, Oeste, Oeste, Oeste, Oeste, Oeste, O…
$ idade        <int> 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 2…
$ sexo         <fct> masculino, masculino, masculino, masculino, masculino, ma…
$ origem       <fct> Alphaville, Alphaville, Alphaville, Alphaville, Alphavill…
$ dente_col    <chr> "ausente_11", "ausente_12", "ausente_13", "ausente_14", "…
$ ausente      <int> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, …
$ dente        <fct> 11, 12, 13, 14, 15, 16, 17, 18, 21, 22, 23, 24, 25, 26, 2…


In [5]:
summary(dados)

   cd_exame         faixa_etaria   regiao_sp         idade      
 Length:80480       13-19:   64   Centro:26176   Min.   : 7.00  
 Class :character   20-44:32608   Leste :20320   1st Qu.:30.00  
 Mode  :character   45-74:43232   Norte :14304   Median :50.00  
                    5-12 :   32   Oeste : 8704   Mean   :48.22  
                    75+  : 4544   Sul   :10976   3rd Qu.:62.00  
                                                 Max.   :93.00  
                                                                
        sexo                origem       dente_col            ausente      
 feminino :48608   Santana     :14272   Length:80480       Min.   :0.0000  
 masculino:31872   Tatuape     :12864   Class :character   1st Qu.:0.0000  
                   Santo Amaro : 9184   Mode  :character   Median :0.0000  
                   Vila Mariana: 8736                      Mean   :0.2241  
                   Jardins     : 8608                      3rd Qu.:0.0000  
                   Jd. A

In [6]:
teste = dados[0:10000,]

In [7]:
# modelo <- glmer(
#     ausente ~ sexo + idade + origem + dente + (1 | cd_exame),
#     data = teste,
#     family = binomial(link = "logit"),
#     control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e5))
#   )

In [ ]:
# modelo

ERROR: Error in eval(expr, envir, enclos): object 'modelo' not found


In [ ]:
# saveRDS(modelo, "/home/aninha/Desktop/Doutorado/Modelos/misto_R/modelo_glmer_sexo_idade_origem_dente.rds")

In [ ]:
# modelo_carregado <- readRDS("/home/aninha/Desktop/Doutorado/Modelos/misto_R/modelo_glmer_sexo_idade_origem_dente.rds")

In [ ]:
# modelo_carregado

In [ ]:
# head(dados)

In [9]:
library(dplyr)

add_tipo_dente <- function(df) {
  df %>%
    mutate(
      tipo_dente = case_when(
        dente %in% c(11, 21, 31, 41) ~ "Central incisor",
        dente %in% c(12, 22, 32, 42) ~ "Lateral incisor",
        
        dente %in% c(13, 23, 33, 43) ~ "Canine",
        
        dente %in% c(14, 24, 34, 44) ~ "First premolar",
        dente %in% c(15, 25, 35, 45) ~ "Second premolar",
        
        dente %in% c(16, 26, 36, 46) ~ "First molar",
        dente %in% c(17, 27, 37, 47) ~ "Second molar",
        dente %in% c(18, 28, 38, 48) ~ "Third molar",
        
        TRUE ~ NA_character_
      ),
      tipo_dente = factor(
        tipo_dente,
        levels = c(
          "Central incisor", "Lateral incisor", "Canine",
          "First premolar", "Second premolar",
          "First molar", "Second molar", "Third molar"
        ),
        ordered = TRUE
      )
    )
}


In [10]:
dados = add_tipo_dente(dados)
head(dados)

cd_exame,faixa_etaria,regiao_sp,idade,sexo,origem,dente_col,ausente,dente,tipo_dente
<chr>,<fct>,<fct>,<int>,<fct>,<fct>,<chr>,<int>,<fct>,<ord>
alp.026080,20-44,Oeste,24,masculino,Alphaville,ausente_11,0,11,Central incisor
alp.026080,20-44,Oeste,24,masculino,Alphaville,ausente_12,0,12,Lateral incisor
alp.026080,20-44,Oeste,24,masculino,Alphaville,ausente_13,0,13,Canine
alp.026080,20-44,Oeste,24,masculino,Alphaville,ausente_14,0,14,First premolar
alp.026080,20-44,Oeste,24,masculino,Alphaville,ausente_15,0,15,Second premolar
alp.026080,20-44,Oeste,24,masculino,Alphaville,ausente_16,0,16,First molar


In [11]:
table(dados$faixa_etaria)


13-19 20-44 45-74  5-12   75+ 
   64 32608 43232    32  4544 

In [12]:
dados_filtrados <- dados %>%
  filter(!faixa_etaria %in% c("13-19", "5-12")) %>% 
  droplevels()      # remove levels não usados

In [ ]:
modelo2 <- glmer(
    ausente ~ sexo + idade + regiao_sp + tipo_dente + (1 | cd_exame),
    data = dados_filtrados,
    family = binomial(link = "logit"),
    control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e5))
  )